In [1]:
import json
import random
import string
import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Download necessary NLTK data
nltk.download("punkt")
nltk.download("wordnet")

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Sample data with medical intents and responses

data = {
    "intents": [
        {"tag": "digestive_issues",
         "patterns": ["I have digestive issues", "What can help with nausea?", "I'm feeling bloated"],
         "responses": ["Ginger Tea: Brew fresh ginger slices in hot water to alleviate nausea and indigestion.",
                       "Apple Cider Vinegar: Mix 1-2 tablespoons of apple cider vinegar in a glass of water to help with digestion and heartburn."]
         },
        {"tag": "cold_and_flu",
         "patterns": ["I have a cold", "What can I take for flu symptoms?", "How to soothe a sore throat?"],
         "responses": ["Honey and Lemon: Mix honey and lemon juice in warm water to soothe sore throats and relieve coughs.",
                       "Steam Inhalation: Inhale steam from a bowl of hot water with a few drops of eucalyptus oil to relieve nasal congestion."]
         },
        {"tag": "skin_care",
         "patterns": ["How to treat sunburn?", "Best remedies for acne", "What can I use for irritated skin?"],
         "responses": ["Aloe Vera Gel: Apply fresh aloe vera gel to sunburned or irritated skin for soothing relief.",
                       "Honey and Cinnamon Mask: Mix honey and cinnamon to create a paste that can help with acne."]
         },
        {"tag": "headaches_and_migraines",
         "patterns": ["I have a headache", "What can help with migraines?", "Tips for relieving headache pain?"],
         "responses": ["Peppermint Oil: Apply diluted peppermint oil to your temples for a cooling sensation and pain relief.",
                       "Cold Compress: Use a cold pack on your forehead or neck to help reduce headache pain."]
         },
        {"tag": "stress_and_anxiety",
         "patterns": ["How to reduce stress?", "What can help with anxiety?", "Natural remedies for relaxation"],
         "responses": ["Chamomile Tea: Drink chamomile tea to promote relaxation and reduce anxiety.",
                       "Lavender Oil: Use lavender essential oil in a diffuser or as a topical application for calming effects."]
         },
        {"tag": "cough_and_throat_irritation",
         "patterns": ["I have a cough", "What to do for throat irritation?", "How to soothe a sore throat?"],
         "responses": ["Salt Water Gargle: Gargle with warm salt water to soothe a sore throat.",
                       "Turmeric Milk: Mix turmeric powder in warm milk for its anti-inflammatory properties.",
                       "Onion Syrup: Chop an onion and mix it with honey. Let it sit overnight, then take a spoonful to help soothe a cough.",
                       "Thyme Tea: Brew thyme leaves in hot water for a few minutes. This can help alleviate cough and bronchial issues."]
         },
        {"tag": "insect_bites_and_allergies",
         "patterns": ["How to relieve insect bites?", "What can I use for allergies?", "I have an allergic reaction"],
         "responses": ["Baking Soda Paste: Mix baking soda with water and apply to insect bites to relieve itching and swelling.",
                       "Oatmeal Bath: Add colloidal oatmeal to a warm bath to soothe irritated skin."]
         },
        {"tag": "sleep_issues",
         "patterns": ["I can't sleep", "What helps with insomnia?", "Tips for better sleep"],
         "responses": ["Warm Milk: Drink warm milk before bedtime to promote sleep.",
                       "Valerian Root Tea: Brew valerian root tea, known for its calming effects."]
         },
        {"tag": "minor_cuts_and_scrapes",
         "patterns": ["How to treat a minor cut?", "What can I use for scrapes?", "How to prevent infection in cuts?"],
         "responses": ["Honey: Apply honey to minor cuts to promote healing due to its antibacterial properties.",
                       "Tea Tree Oil: Dilute tea tree oil and apply it to cuts to help prevent infection."]
         },
        {"tag": "hair_care",
         "patterns": ["How to improve hair health?", "What natural remedies can I use for my hair?", "Tips for shiny hair"],
         "responses": ["Coconut Oil: Use coconut oil as a hair mask for hydration and shine.",
                       "Apple Cider Vinegar Rinse: Dilute apple cider vinegar with water and use it as a hair rinse to promote shine and scalp health."]
         },
        {"tag": "joint_pain_and_inflammation",
         "patterns": ["I have joint pain", "What can help with arthritis?", "Natural remedies for inflammation"],
         "responses": ["Turmeric and Ginger Tea: Brew together for relief from joint pain.",
                       "Epsom Salt Bath: Soak in a warm bath with Epsom salt to help ease muscle aches and joint pain."]
         },
        {"tag": "allergies_and_sinus_issues",
         "patterns": ["I have allergies", "What can help with sinus issues?", "How to relieve sinus pressure?"],
         "responses": ["Neti Pot: Use a neti pot with a saline solution to rinse out your nasal passages and alleviate sinus pressure.",
                       "Local Honey: Consuming local honey may help your body adapt to allergens in your environment."]
         },
        {"tag": "earache",
         "patterns": ["I have an earache", "What can I do for ear pain?", "How to soothe ear pain?"],
         "responses": ["Warm Olive Oil: Place a few drops of warm (not hot) olive oil in the ear to help soothe the pain.",
                       "Garlic Oil: Crush garlic and steep it in warm olive oil. Strain and use as ear drops."]
         },
        {"tag": "toothache",
         "patterns": ["I have a toothache", "How to relieve tooth pain?", "What can help with a dental ache?"],
         "responses": ["Clove Oil: Dab a bit of clove oil on a cotton ball and apply it to the aching tooth for relief.",
                       "Saltwater Rinse: Gargle with warm salt water to reduce swelling and kill bacteria."]
         },
        {"tag": "foot_odor",
         "patterns": ["I have foot odor", "What can I do for smelly feet?", "How to reduce foot odor?"],
         "responses": ["Baking Soda Soak: Soak your feet in a mixture of water and baking soda to neutralize odors.",
                       "Vinegar Soak: Soaking your feet in a diluted vinegar solution can help combat bacteria and odors."]
         },
        {"tag": "hiccups",
         "patterns": ["I have hiccups", "How to get rid of hiccups?", "What can I do to stop hiccups?"],
         "responses": ["Sugar: Swallow a spoonful of sugar to reset the nerves controlling your diaphragm.",
                       "Cold Water: Sip cold water slowly to stop the hiccups."]
         },
        {"tag": "nausea_and_motion_sickness",
         "patterns": ["I feel nauseous", "What can help with motion sickness?", "Tips for relieving nausea"],
         "responses": ["Peppermint Tea: Peppermint has calming properties that can help reduce nausea.",
                       "Lemon Water: Sipping lemon water can also help reduce nausea."]
         },
        {"tag": "menstrual_cramps",
         "patterns": ["I have menstrual cramps", "How to relieve period pain?", "What can help with cramping?"],
         "responses": ["Heat Pack: Applying heat to your lower abdomen can help soothe cramps.",
                       "Ginger Tea: Ginger helps with pain relief and reducing inflammation during menstruation."]
         },
        {"tag": "dry_skin",
         "patterns": ["I have dry skin", "What can I do for itchy skin?", "How to hydrate dry skin?"],
         "responses": ["Coconut Oil: Apply coconut oil to dry skin for intense hydration.",
                       "Oatmeal Bath: Taking an oatmeal bath helps to soothe itchy, dry skin."]
         },
        {"tag": "sunburn",
         "patterns": ["I have sunburn", "What can help with sunburn?", "How to treat a sunburn?"],
         "responses": ["Cucumber Slices: Placing cucumber slices on the affected area can provide cooling relief.",
                       "Aloe Vera: Aloe vera gel is one of the best remedies for sunburn, providing cooling and healing properties."]
         },
        {"tag": "cough_relief",
         "patterns": ["How to soothe a cough?", "What can I take for a cough?", "I have a persistent cough"],
         "responses": ["Onion Syrup: Chop an onion and mix it with honey. Let it sit overnight, then take a spoonful to help soothe a cough.",
                       "Thyme Tea: Brew thyme leaves in hot water for a few minutes. This can help alleviate cough and bronchial issues."]
         },
        {"tag": "nasal_congestion",
         "patterns": ["I have nasal congestion", "What can help clear my sinuses?", "How to relieve blocked nose?"],
         "responses": ["Peppermint Steam: Inhale steam from hot water with a few drops of peppermint essential oil to clear nasal passages.",
                       "Spicy Foods: Eating spicy foods can temporarily open nasal passages and help relieve congestion."]
         },
        
        {"tag": "symptom_check",
         "patterns": ["I have a headache", "I feel nauseous", "I'm dizzy", "I have a sore throat"],
         "responses": ["Have you tried drinking plenty of water?", 
                       "Resting in a dark room may help.", 
                       "You might want to consult a doctor if symptoms persist."]
         },
        {"tag": "remedy",
         "patterns": ["What can I take for a cold?", "Suggest a remedy for headache", "How to treat nausea?", "Best way to relieve stress"],
         "responses": ["For a cold, try drinking warm fluids and resting.",
                       "For headaches, over-the-counter pain relief may be effective.",
                       "Ginger tea can help with nausea.",
                       "Meditation and deep breathing exercises can reduce stress."]
         },
        {"tag": "diet_advice",
         "patterns": ["What should I eat for energy?", "Foods good for health", "Best diet for weight loss"],
         "responses": ["Include whole grains, fruits, and lean proteins in your diet.",
                       "Eating vegetables and fruits daily is essential for health.",
                       "Consult a nutritionist for personalized diet plans."]
         },
        {"tag": "general_advice",
         "patterns": ["How can I stay healthy?", "Tips for good health", "How to boost immunity?"],
         "responses": ["Stay hydrated and exercise regularly.",
                       "Ensure you get enough sleep and manage stress.",
                       "Eating a balanced diet helps boost immunity."]
         },
        {"tag": "goodbye",
         "patterns": ["bye", "g2g", "see ya", "adios", "cya"],
         "responses": ["Take care! Stay healthy!", "Goodbye! Stay safe!", "See you later!"]
         }
    ]
}

[nltk_data] Downloading package punkt to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Initialize words, classes, doc_x, and doc_y
words = []
classes = []
doc_x = []
doc_y = []

# Tokenize and process the data
for intent in data["intents"]:
    for pattern in intent["patterns"]:
        tokens = nltk.word_tokenize(pattern)
        words.extend(tokens)
        doc_x.append(pattern)
        doc_y.append(intent["tag"])
    if intent["tag"] not in classes:
        classes.append(intent["tag"])

# Process the words list, removing punctuation
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in string.punctuation]
words = sorted(set(words))
classes = sorted(set(classes))

# Preparing the training data
training = []
out_empty = [0] * len(classes)
for idx, doc in enumerate(doc_x):
    bow = []
    text = lemmatizer.lemmatize(doc.lower())
    for word in words:
        bow.append(1) if word in text else bow.append(0)
    output_row = list(out_empty)
    output_row[classes.index(doc_y[idx])] = 1
    training.append([bow, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

# Split the training data into input (X) and output (y)
train_X = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

# Create a Sequential model
model = Sequential()
model.add(Dense(128, input_shape=(len(train_X[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Create the Adam optimizer with a specified learning rate
adam = tf.keras.optimizers.Adam(learning_rate=0.01)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

print(model.summary())

# Train the model
model.fit(x=train_X, y=train_y, epochs=500, verbose=1)

# Function to clean user input text
def clean_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# Function to convert user input into a bag-of-words representation
def bag_of_words(text, vocab):
    tokens = clean_text(text)
    bow = [0] * len(vocab)
    for w in tokens:
        for idx, word in enumerate(vocab):
            if word == w:
                bow[idx] = 1
    return np.array(bow)

# Function to check for specific keywords
def check_keywords(text):
    keywords = ["tired", "fatigued", "exhausted", "drained"]
    for keyword in keywords:
        if keyword in text.lower():
            return True
    return False

# Function to predict the user's intent based on their input
def pred_class(text, vocab, labels):
    bow = bag_of_words(text, vocab)
    result = model.predict(np.array([bow]))[0]
    thresh = 0.2
    y_pred = [[idx, res] for idx, res in enumerate(result) if res > thresh]
    y_pred.sort(key=lambda x: x[1], reverse=True)

    if y_pred:
        return_list = []
        for r in y_pred:
            return_list.append(labels[r[0]])
        
        # Check for specific keywords to modify the response
        if check_keywords(text):
            return ["tired"]  # This is a special case for tiredness
        return return_list
    return []

# Function to get a response from the predicted intent
def get_response(intents_list, intents_json):
    if "tired" in intents_list:
        return "It's important to rest! Make sure you're taking care of yourself. If you need tips on relaxation or ways to recharge, feel free to ask!"

    tag = intents_list[0]
    list_of_intents = intents_json["intents"]
    for i in list_of_intents:
        if i["tag"] == tag:
            result = random.choice(i["responses"])
            break
    return result

C:\Users\maria selciya\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        16,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 27)             │         1,755 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,011 (101.61 KB)

 Trainable params: 26,011 (101.61 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.1076 - loss: 3.3297
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1487 - loss: 3.1992
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1507 - loss: 3.1055 
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2153 - loss: 2.9966  
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2094 - loss: 2.9161
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3307 - loss: 2.6832
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4305 - loss: 2.4775 
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4892 - loss: 2.1124
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4990 - loss: 2.0600
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5186 - loss: 1.7394 
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6066 - loss: 1.5060 
Epoch 12/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6477 - lo

In [ ]:
while True:
    message = input("")
    
    # Break condition: if user types exit, quit, or bye
    if message.lower() in ["exit", "quit", "bye","catch you later","i'm leaving"," i need to go"]:
        print("Goodbye!! Have a nice day....Stay Super positive!!!")
        break

    # Use pred_class to get intents and respond
    intents = pred_class(message, words, classes)
    result = get_response(intents, data)
    print(result)